# Lab 1 — From a vague concern to defensible evidence

**CS203 · Data Science Fundamentals · New Uzbekistan University**  
**STUDENT COPY · Individualised notebook**  
**Duration:** 2 hours · **Reading:** Chapters 1–2 · **Submit:** one reproducible notebook

## What this lab is really about

This is not a “write some pandas code” exercise. A colleague gives you a vague concern. You will turn it into an analytical question that can be answered with the available data, decide what one row must mean, build only the table you need, inspect a limitation, and write a cautious response.

The data are synthetic, not real student records. That lets us practise responsible analysis without pretending that a technically possible intervention is automatically justified.

**Learning goals**

- translate a vague claim into a precise, answerable descriptive question;
- name the unit, population, numerator/denominator, time window, and comparison before calculating;
- join data safely and prove that your final table has the intended unit;
- report evidence, uncertainty, and one meaningful limitation; and
- distinguish a group-level audit from an automated decision about a person.


## Two-hour route

| Time | Activity | What you produce |
| --- | --- | --- |
| 0:00–0:10 | Set up, enter your ID, receive your individual brief | Assignment fingerprint |
| 0:10–0:25 | Shared data contract warm-up | Unit/key notes |
| 0:25–0:40 | Turn the vague brief into a question | Question contract + plan |
| 0:40–1:15 | Build the individual analysis | Safe analytical table + evidence table |
| 1:15–1:35 | Inspect shape, group size, and a data-quality threat | Figure + audit |
| 1:35–1:50 | Peer challenge and instructor/TA checkpoint | Revised claim |
| 1:50–2:00 | Write the decision memo and restart-run check | Final notebook |

### Scaffold balance

Exactly **9 of the 15 code cells (60%)** are deliberately provided: loading data, assigning a reproducible individual case, safe helper code, a common early-week table, and the submission structure. You complete the remaining **6 code cells (40%)**, plus the question, design choices, validation, interpretation, peer response, and decision memo.

Everyone completes the shared opening. After the assignment cell, each student has a different case and a different deterministic sample. Your final fingerprint must stay visible in the submitted notebook.


## Academic integrity and responsible use of AI

You may use documentation or an AI assistant to understand a Python error or syntax **after** you have written your own plan. Do not ask a tool to choose your question, make your interpretation, or write your final memo. If you use any AI assistance, complete the declaration at the end: name the tool, the narrow task, and what you changed after checking it.

Generic copied code or prose will usually not fit your assigned case, sample, fingerprint, and follow-up explanation. Marks reward the quality of your reasoning and evidence, not only whether code runs.


## 0. Setup — run the next three cells in order


In [ ]:
from pathlib import Path
import hashlib
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# In Colab, uncomment these two lines once if your Drive is not already mounted.
# if IN_COLAB:
#     drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/fds/data") if IN_COLAB else Path.cwd() / "data"
RAW_DATA_BASE = (
    "https://raw.githubusercontent.com/Sabokrou/Sabokrou.github.io/main/"
    "teaching/data-science-fundamentals/datasets"
)

def load_course_table(filename: str) -> pd.DataFrame:
    # Load a local course-data copy when available; otherwise use the released Lab 1 data.
    local_file = DATA / filename
    if local_file.exists():
        return pd.read_csv(local_file)
    print(f"Using the Lab 1 data source for {filename}.")
    return pd.read_csv(f"{RAW_DATA_BASE}/{filename}")


students = load_course_table("students.csv")
enrolments = load_course_table("enrolments.csv")
activity = load_course_table("activity.csv")
outcomes = load_course_table("outcomes.csv")

print("Loaded:")
for name, table in {
    "students": students,
    "enrolments": enrolments,
    "activity": activity,
    "outcomes": outcomes,
}.items():
    print(f"  {name:<12} {table.shape[0]:>6,} rows × {table.shape[1]} columns")


In [ ]:
# Required: use your official student ID exactly as it appears in HERO.
# It is used only inside this notebook to select a reproducible case and data slice.
STUDENT_CODE = "REPLACE_WITH_YOUR_STUDENT_ID"

if STUDENT_CODE == "REPLACE_WITH_YOUR_STUDENT_ID":
    raise ValueError("Replace STUDENT_CODE with your official student ID before continuing.")
if not re.fullmatch(r"[A-Za-z0-9_-]{4,32}", STUDENT_CODE):
    raise ValueError("Use 4–32 letters, digits, underscores, or hyphens; do not enter your name.")

CASE_CARDS = [
    {
        "code": "A", "title": "Part-time experience",
        "brief": "The Academic Office says: ‘Part-time students may be struggling. Should we change how support is offered?’",
        "decision": "Decide whether the evidence justifies a follow-up investigation or a changed support offer — not an automatic decision about any individual.",
        "focus": "study mode and student-level course outcomes",
        "minimum_tables": ["students", "enrolments"],
        "caution": "A difference between groups does not show why it exists. Give every student equal weight before comparing student outcomes.",
    },
    {
        "code": "B", "title": "Early engagement and withdrawal",
        "brief": "Student Support says: ‘Students who later leave might not be engaging in the first weeks. Is that a useful concern?’",
        "decision": "Decide whether the descriptive pattern is strong enough to motivate a human-led support conversation, not a prediction or sanction rule.",
        "focus": "weeks 1–4 engagement and later withdrawal",
        "minimum_tables": ["students", "outcomes"],
        "caution": "Use only early-week information. A later withdrawal date or full-term activity would leak information unavailable at the time of support.",
    },
    {
        "code": "C", "title": "Can the satisfaction survey speak for everyone?",
        "brief": "The Teaching Committee says: ‘The satisfaction average is low. Can we report it as the experience of all students?’",
        "decision": "Decide whether the respondent-only number is an adequate description of the whole assigned cohort.",
        "focus": "survey response, early engagement, and non-response",
        "minimum_tables": ["students", "outcomes"],
        "caution": "A mean among respondents answers a question about respondents. It does not automatically answer a question about all students.",
    },
    {
        "code": "D", "title": "Programme pattern",
        "brief": "The Faculty Office says: ‘One programme may have a retention problem. Should we claim that it does?’",
        "decision": "Decide whether the observed programme-level pattern is large and stable enough to justify a closer qualitative investigation.",
        "focus": "programme and withdrawal",
        "minimum_tables": ["students", "outcomes"],
        "caution": "Programme differences are descriptive. They may reflect intake, course design, workload, or unmeasured context; do not make a causal claim.",
    },
    {
        "code": "E", "title": "First-generation equity audit",
        "brief": "An equity group asks: ‘Are outcomes similar for first-generation and continuing-generation students?’",
        "decision": "Decide whether an outcome gap warrants a review of the learning environment or support access.",
        "focus": "first-generation status and withdrawal",
        "minimum_tables": ["students", "outcomes"],
        "caution": "This is an audit of group-level outcomes. Never use first-generation status to allocate, deny, or automate support for an individual.",
    },
    {
        "code": "F", "title": "Entry score and persistence",
        "brief": "An admissions colleague says: ‘Entry scores may explain who withdraws. Should we act on that?’",
        "decision": "Decide whether the observed association is informative enough to study further, while rejecting any unsupported individual-level rule.",
        "focus": "entry score and withdrawal",
        "minimum_tables": ["students", "outcomes"],
        "caution": "An association is not a cause and does not prove that an entry-score threshold would be fair or useful.",
    },
    {
        "code": "G", "title": "Platform access and study mode",
        "brief": "A digital-learning colleague says: ‘Part-time students may have less early access to the platform. Is that visible in the data?’",
        "decision": "Decide whether the pattern is large enough to prompt a check of access, schedule, or course-design barriers.",
        "focus": "study mode and early platform engagement",
        "minimum_tables": ["students", "outcomes"],
        "caution": "Logins and minutes are proxies. They do not measure motivation, learning quality, or a student’s circumstances directly.",
    },
    {
        "code": "H", "title": "Which pass-rate denominator?",
        "brief": "The Registrar says: ‘What is the pass-rate difference for the two study modes?’ The statement does not say whether a row means a person or a course registration.",
        "decision": "Decide which denominator answers the Registrar’s stated purpose and show how the conclusion changes under the other valid denominator.",
        "focus": "student-level versus registration-level pass rates",
        "minimum_tables": ["students", "enrolments"],
        "caution": "Both denominators can be correct, but they answer different questions. Name the unit before you calculate a percentage.",
    },
]
assignment_seed = int.from_bytes(
    hashlib.blake2b(STUDENT_CODE.encode("utf-8"), digest_size=8).digest(), "big"
)
rng = np.random.default_rng(assignment_seed)
sample_size = 320 + int(assignment_seed % 81)  # 320–400 students; varies by ID
assigned_ids = np.sort(
    rng.choice(students["student_id"].to_numpy(), size=sample_size, replace=False)
)
case_card = CASE_CARDS[assignment_seed % len(CASE_CARDS)]
sample_checksum = hashlib.blake2b(
    ",".join(map(str, assigned_ids)).encode("utf-8"), digest_size=3
).hexdigest().upper()
ASSIGNMENT_FINGERPRINT = f"{case_card['code']}-{sample_size}-{sample_checksum}"

print("YOUR INDIVIDUAL ASSIGNMENT")
print("Fingerprint:", ASSIGNMENT_FINGERPRINT)
print("Case:", case_card["title"])
print("Vague briefing:", case_card["brief"])
print("Decision context:", case_card["decision"])
print("Focus:", case_card["focus"])
print("Minimum tables to consider:", ", ".join(case_card["minimum_tables"]))
print("Important caution:", case_card["caution"])


In [ ]:
def assigned_only(table: pd.DataFrame) -> pd.DataFrame:
    return table.loc[table["student_id"].isin(assigned_ids)].copy()

students_assigned = assigned_only(students)
enrolments_assigned = assigned_only(enrolments)
activity_assigned = assigned_only(activity)
outcomes_assigned = assigned_only(outcomes)

assigned_shapes = pd.DataFrame(
    {
        "table": ["students", "enrolments", "activity", "outcomes"],
        "rows": [
            len(students_assigned),
            len(enrolments_assigned),
            len(activity_assigned),
            len(outcomes_assigned),
        ],
        "one row means": [
            "one student",
            "one student-course registration",
            "one student-week",
            "one student",
        ],
    }
)
assigned_shapes


In [ ]:
# Provided orientation: inspect names and data types before you choose a variable.
for table_name, table in {
    "students": students_assigned,
    "enrolments": enrolments_assigned,
    "activity": activity_assigned,
    "outcomes": outcomes_assigned,
}.items():
    print(f"\n{table_name.upper()}")
    print(table.dtypes.to_string())


## 1. Shared warm-up: a data contract before a calculation (0:10–0:25)

The same `student_id` can appear once, several times, or once per week depending on the table. That is not a nuisance; it defines what a percentage or average means.

Run the audit. Then complete the short table below **in your own words**. You may consult `DATA_DICTIONARY.md`, but do not copy it without understanding it.

| Table | What does one row mean? | Candidate key | Why a direct join could mislead |
| --- | --- | --- | --- |
| `students` | _replace_ | _replace_ | _replace_ |
| `enrolments` | _replace_ | _replace_ | _replace_ |
| `activity` | _replace_ | _replace_ | _replace_ |
| `outcomes` | _replace_ | _replace_ | _replace_ |

**Warm-up response (2–3 sentences):** The Faculty Office asks, “What percentage of students are part-time?” Explain why counting rows in `enrolments` would not answer that exact question.

> _Write here._


In [ ]:
def key_audit(table: pd.DataFrame, columns: list[str]) -> dict:
    # Return evidence about whether the stated columns form a key.
    duplicate_rows = int(table.duplicated(columns).sum())
    return {
        "candidate_key": ", ".join(columns),
        "rows": len(table),
        "duplicate_key_rows": duplicate_rows,
        "passes": duplicate_rows == 0,
    }

contract_audit = pd.DataFrame(
    [
        {"table": "students", **key_audit(students_assigned, ["student_id"])},
        {"table": "enrolments", **key_audit(enrolments_assigned, ["student_id", "course_code", "term"])},
        {"table": "activity", **key_audit(activity_assigned, ["student_id", "iso_week"])},
        {"table": "outcomes", **key_audit(outcomes_assigned, ["student_id"])},
    ]
)
contract_audit


In [ ]:
def checked_merge(left: pd.DataFrame, right: pd.DataFrame, *, on: str, validate: str, how: str = "left") -> pd.DataFrame:
    # A supplied helper: students still choose the tables, unit, and cardinality.
    before = len(left)
    merged = left.merge(right, on=on, how=how, validate=validate)
    print(f"{before:,} rows before merge → {len(merged):,} rows after merge ({validate})")
    return merged


print("Use checked_merge only after you can explain why the stated cardinality is correct.")


In [ ]:
# This is a safe common scaffold: it uses only weeks 1–4.
# It deliberately does NOT use full-term activity, which would leak later information.
early_activity = (
    activity_assigned.loc[activity_assigned["iso_week"].between(1, 4)]
    .groupby("student_id", as_index=False)
    .agg(
        early_logins_from_activity=("logins", "sum"),
        early_minutes_from_activity=("minutes_on_platform", "sum"),
        early_submissions_from_activity=("submissions", "sum"),
    )
)
assert early_activity["student_id"].is_unique
early_activity.head()


In [ ]:
def group_snapshot(frame: pd.DataFrame, group: str, value: str) -> pd.DataFrame:
    # A compact scaffold for reporting group size alongside a numerical summary.
    return (
        frame.groupby(group, dropna=False)
        .agg(n=("student_id", "size"), mean=(value, "mean"), median=(value, "median"))
        .reset_index()
    )


print("The helper reports n, mean, and median. You must still decide whether those are appropriate for your question.")


## 2. Frame the brief as an analytical question (0:25–0:40)

Read your assigned vague briefing again. It is intentionally not yet a usable question. Before you code, write a question contract. A strong contract can be checked by someone who has not seen your notebook.

**Do not use causal language** such as “causes,” “effect,” or “because” unless the data-generating design supports it. This dataset supports descriptive comparisons and audits, not causal proof.

### Your question contract

- **Vague concern in one sentence:** _replace_
- **Decision the analysis should inform (not automate):** _replace_
- **Precise analytical question (one sentence):** _replace_
- **Population and inclusion rule:** _replace_
- **Unit of observation in your final table:** _replace_
- **Comparison, event, or quantity:** _replace_
- **Numerator and denominator (if reporting a rate/share):** _replace_
- **Time window:** _replace_
- **One thing the data cannot tell you:** _replace_

### Plan before code

In 3–5 bullets, state which tables you will use, which table you will start from, any aggregation you need before a join, the join cardinality you expect, and one check that could falsify your plan.

> _Write here._


## 3. Build evidence for your individual question (0:40–1:15)

Work from your plan. You are not required to reproduce a secret “correct” analysis; you are required to make an analysis that matches your stated question and survives checks.

**Non-negotiable rules**

- A student-level question needs one row per student in the final table.
- Aggregate repeated registration or weekly activity rows before joining to a student-level table.
- Use `validate=` in every merge and inspect the number of rows before/after.
- Never use `withdrew_week` or weeks 5–14 activity to answer an early-support question.
- For sensitive attributes, audit group-level patterns only; do not create a rule for individuals.


In [ ]:
# TODO 1 — Build ONE analytical table that matches the unit in your own question.
# Expected: about 8–15 lines. Save it as `analysis_frame`.
#
# Rules:
#   • If you use enrolments or activity, aggregate BEFORE joining to student-level data.
#   • Use merge(..., validate=...) whenever you join.
#   • Do not use `withdrew_week` or full-term activity as an early-support feature.
#   • Keep `student_id` so that you can test the final unit.
#
# Useful supplied objects: students_assigned, enrolments_assigned,
# activity_assigned, outcomes_assigned, early_activity.

pass


**Unit check interpretation (one sentence):** _Write what one row in `analysis_frame` represents, and cite the check you ran._


In [ ]:
# TODO 2 — Write your evidence that `analysis_frame` has the unit you claimed.
# Include at least two checks. One must test that student_id is unique if your
# question is student-level. If it is registration-level, explicitly check that instead.
# Add a one-sentence interpretation immediately below this cell.

pass


In [ ]:
# TODO 3 — Create your main evidence table.
# It must show the group/category, the denominator n, and a statistic or rate that
# answers your question. A mean alone is not enough when n differs or a distribution is skewed.
# Name the result `evidence_table` and display it.

pass


**Evidence interpretation (3–4 sentences):** State the main pattern with the group sizes and measurement unit. Then state what a reader must *not* conclude from this table.

> _Write here._


## 4. Inspect, challenge, and qualify the result (1:15–1:35)


In [ ]:
# TODO 4 — Make ONE figure that helps a reader inspect your evidence.
# Use a message title, label the unit, and do not hide important group sizes.
# Then write 2–3 sentences below explaining what the figure supports and what it cannot prove.

pass


**Figure interpretation (2–3 sentences):** _Write here. Include a reason the chart is appropriate for this question and a limitation._

A figure is not decoration. It should help a reader check whether a single reported statistic hides a group-size, spread, or shape issue.


In [ ]:
# TODO 5 — Audit a relevant missingness, selection, or measurement issue.
# For most cases, inspect midterm_satisfaction response by a group relevant to your question.
# For Case C, response/non-response is central and you must compare respondents with non-respondents.
# Report both a count and a percentage, then write a limitation below.

pass


**Quality limitation (2–3 sentences):** _Write here. Name the missingness/selection/proxy issue, say why it matters for your exact claim, and avoid claiming that the issue is solved merely because you noticed it._


In [ ]:
# TODO 6 — Do one robustness / alternative-definition check.
# Examples: compare mean with median; repeat a rate with a different valid denominator;
# report group counts before making a comparison; or state why an alternative would change
# the decision. Save a compact result or calculation as `robustness_check`.

pass


## 5. Peer challenge and checkpoint (1:35–1:50)

Exchange notebooks with a nearby student for five minutes. Do not solve each other’s code. The reviewer should ask:

1. Does the stated question match the final table’s unit?
2. Is the denominator visible and appropriate?
3. What alternative explanation or data issue could change the claim?

**Peer reviewer’s first name or initials:** _replace_  
**One challenge they raised:** _replace_  
**What I revised after the challenge:** _replace_

At the instructor/TA checkpoint, show your fingerprint, question contract, `analysis_frame` shape, and evidence table. Be ready to explain one design decision without reading from the notebook.


## 6. Final decision memo (1:50–2:00)

Write **150–220 words** to the person in your assigned brief. Your memo must include:

- the question and the unit/denominator;
- the central numerical evidence and group size(s);
- a cautious conclusion for the stated decision;
- one limitation or alternative explanation; and
- one responsible next step (for example, a qualitative check, data-collection improvement, or support review).

Do not write “the data prove…” or recommend an automated decision about a person.

> _Write your memo here._


In [ ]:
# Optional self-check after you write your memo above.
final_memo = ''  # Copy your finished memo here only if you want the word-count check.
if final_memo.strip():
    n_words = len(final_memo.split())
    print(f"Memo word count: {n_words}")
    if not 150 <= n_words <= 220:
        print("Revise: the target is 150–220 words.")
else:
    print("Write the memo in the markdown cell above; this word-count check is optional.")

print("Submission fingerprint:", ASSIGNMENT_FINGERPRINT)


## Submission checklist and marking guide

Before submitting, **Restart kernel → Run all**. Your submission is incomplete if it only works because cells were run out of order.

- [ ] My official-ID-based fingerprint is visible.
- [ ] My question contract specifies unit, population, time, and denominator where needed.
- [ ] `analysis_frame` and key/join checks match the question.
- [ ] I show group sizes, evidence, and one meaningful figure.
- [ ] I discuss a relevant limitation and an alternative/robustness check.
- [ ] I included the peer challenge and final decision memo.
- [ ] If I used AI, I completed the declaration below.

| Criterion | Weight | What earns credit |
| --- | ---: | --- |
| Framing and analytical design | 30% | Question is answerable; unit/denominator/time are explicit; plan matches decision. |
| Reproducible data work | 25% | Safe aggregation/joins, validation, readable code, restart-run works. |
| Evidence and communication | 25% | Correct evidence, group sizes, useful visual, memo answers the actual question. |
| Critical reflection | 15% | Honest limitation, robustness/alternative definition, responsible interpretation. |
| Process and integrity | 5% | Fingerprint, peer challenge, and transparent AI declaration. |

### AI-use declaration

- **Tool used (or “none”):** _replace_
- **Narrow task it helped with:** _replace_
- **What I checked, changed, or rejected:** _replace_
